# Three Energy-Level Views Across Databases

This notebook generates publication-ready energy-level distribution plots for a selected ion across two or three atomic databases. It uses the grouping utilities in `plot_tbl_cflev.py` and can produce the following PDF files:

1. `Evsdb_1_full_no_labels.pdf`: the full energy range without category labels.
2. `Evsdb_2_low_with_labels.pdf`: energies below the selected boundary with category labels.
3. `Evsdb_3_high_with_labels.pdf`: energies above the selected boundary with category labels.
4. `Evsdb_0_three_views_combined.pdf`: an optional horizontal combination of the three views.

Energy intervals with no levels in any database can be compressed for readability. `compress_gap_threshold` sets the minimum physical gap to compress, while `compressed_gap_display` sets its displayed length. Axis labels continue to show the physical energy values.


In [ ]:
# Analysis and plotting parameters
ion = '020807'
dbnames = ['atomdb313', 'chianti1102', 'spex30801']
unit = 'eV'

# Boundary separating the low- and high-energy views.
energy_boundary = 1000

# Compress a shared empty interval when its physical width exceeds the threshold.
cut_big_gap = True
compress_gap_threshold = 5.0
compressed_gap_display = 3.0

# Physical energy interval between y-axis tick labels; use None for automatic ticks.
ytick_interval = 20000

# Draw break marks at compressed intervals.
draw_compress_breaks = True
break_mark_width = 0.035
break_mark_height = 0.012

# Figure dimensions.
figsize_full = (4, 7)
figsize_split = (5, 6.5)
figsize_combined = (15, 5.5)

shadow_alpha = 0.1
line_width = 0.8
n_colors = 25
dpi = 200
show_boundary_line = True
output_format = 'pdf'


In [ ]:
# Import dependencies and load the plotting module from the project package.
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from plot import plot_tbl_cflev as plotmod


In [ ]:
# Read the cleaned level tables and assign configuration categories.
def prepare_energy_data(ion, dbnames, unit):
    """Load level data, derive category fields, and create the output directory."""
    cfg_part, set_name, lsort_name = plotmod.seq_ele_sort(int(ion[:2]))
    if cfg_part is None:
        raise ValueError(f'Sequence {ion[:2]} is not defined in seq_ele_sort')

    lenergy = []
    for i, dbname in enumerate(dbnames):
        file_path_atdata = os.path.join(plotmod.directory, 'clean_data', ion + '.h5')
        dataset_path = '/lev_' + dbname
        print('Reading:', dataset_path)
        data = plotmod.show_tbl_lev.lev_df(file_path_atdata, dataset_path)

        split_data = data['cfg'].astype(str).str.split('.', expand=True)
        need_cols = len(cfg_part)
        have_cols = split_data.shape[1]
        if have_cols < need_cols:
            for col_i in range(have_cols, need_cols):
                split_data[col_i] = None
        if have_cols > need_cols:
            split_data = split_data.iloc[:, :need_cols]
        split_data.columns = cfg_part
        data[cfg_part] = split_data

        data['cfg_c'] = data['cfg'].apply(plotmod.clean_cfg_sha)
        data = data.rename(columns={'idx': 'idx' + str(i + 1), unit: unit + str(i + 1)})
        lenergy.append(data)

    sum_table = plotmod.levelnum_table(dbnames, lenergy, cfg_part, set_name, lsort_name, unit)
    try:
        sum_table = sum_table.drop('last_char', axis=1)
    except KeyError:
        pass

    output_dir = os.path.join(
        plotmod.directory,
        'data',
        'seq' + ion[:2],
        ion,
        '_'.join(dbnames),
        'three_energy_views',
    )
    os.makedirs(output_dir, exist_ok=True)
    return lenergy, sum_table, output_dir


In [ ]:
# Plotting functions for the three energy views.
def _get_category_data(data, category, sum_table, col):
    # Select levels directly by category name to keep categories aligned.
    if category == 'other':
        known_categories = set(sum_table.loc[sum_table['category'] != 'total', 'category'])
        known_categories.discard('other')
        cat_data = data.loc[~data['cfg_c'].isin(known_categories)].copy()
    else:
        cat_data = data.loc[data['cfg_c'] == category].copy()
    return cat_data.sort_values(by=col).reset_index(drop=True)


def _all_energy_values(dbnames, lenergy, unit):
    vals = []
    for i in range(len(dbnames)):
        col = unit + str(i + 1)
        vals.extend(pd.to_numeric(lenergy[i][col], errors='coerce').dropna().tolist())
    return np.array(vals, dtype=float)


def _collect_visible_intervals(dbnames, lenergy, sum_table, unit, view_min, view_max):
    intervals = []
    for i, dname in enumerate(dbnames):
        data = lenergy[i]
        col = unit + str(i + 1)
        for j in range(len(sum_table) - 1):
            if int(sum_table.loc[j, dname]) == 0:
                continue
            category = sum_table.loc[j, 'category']
            cat_data = _get_category_data(data, category, sum_table, col)
            ys = pd.to_numeric(cat_data[col], errors='coerce').dropna()
            if len(ys) == 0:
                continue
            y0, y1 = float(ys.min()), float(ys.max())
            y0, y1 = max(y0, view_min), min(y1, view_max)
            if y0 <= y1:
                intervals.append((y0, y1))
    return intervals


def _make_gap_compressor(dbnames, lenergy, sum_table, unit, view_min, view_max,
                         cut_big_gap=True, gap_threshold=5.0, display_gap=2.0,
                         compress_outer_gaps=True):
    """Create a coordinate transform for shared empty energy intervals.

    ``gap_threshold`` is the minimum physical width to compress, and
    ``display_gap`` is the displayed width after compression.
    """
    def identity(y):
        if hasattr(y, 'apply'):
            return y
        if isinstance(y, (list, tuple, np.ndarray)):
            return y
        return y

    if (not cut_big_gap) or gap_threshold is None or display_gap is None:
        return identity, []

    intervals = _collect_visible_intervals(dbnames, lenergy, sum_table, unit, view_min, view_max)
    if len(intervals) == 0:
        return identity, []

    intervals.sort(key=lambda x: x[0])
    merged = []
    for y0, y1 in intervals:
        if not merged or y0 > merged[-1][1]:
            merged.append([y0, y1])
        else:
            merged[-1][1] = max(merged[-1][1], y1)

    steps = []
    if compress_outer_gaps and merged[0][0] - view_min > gap_threshold:
        steps.append((view_min, merged[0][0], merged[0][0] - view_min, display_gap))

    for k in range(len(merged) - 1):
        gap_start = merged[k][1]
        gap_end = merged[k + 1][0]
        gap = gap_end - gap_start
        if gap > gap_threshold:
            steps.append((gap_start, gap_end, gap, display_gap))

    if compress_outer_gaps and view_max - merged[-1][1] > gap_threshold:
        steps.append((merged[-1][1], view_max, view_max - merged[-1][1], display_gap))

    if len(steps) == 0:
        return identity, []

    def compress_scalar(y):
        if pd.isna(y):
            return y
        y = float(y)
        shift = 0.0
        for gap_start, gap_end, gap, display_len in steps:
            if y <= gap_start:
                break
            if y < gap_end:
                # Map values linearly within a compressed interval.
                return gap_start - shift + (y - gap_start) * display_len / gap
            shift += gap - display_len
        return y - shift

    def compress(y):
        if hasattr(y, 'apply'):
            return y.apply(compress_scalar)
        if isinstance(y, np.ndarray):
            return np.array([compress_scalar(v) for v in y])
        if isinstance(y, (list, tuple)):
            return [compress_scalar(v) for v in y]
        return compress_scalar(y)

    return compress, steps


def _make_real_energy_ticks(view_min, view_max, tick_interval=None, steps=None, nticks=8):
    # Generate physical-energy ticks and omit intermediate ticks inside compressed gaps.
    steps = steps or []
    eps = max(abs(view_max - view_min), 1.0) * 1e-10

    if view_max <= view_min:
        ticks = [float(view_min)]
    elif tick_interval is None or tick_interval <= 0:
        ticks = list(np.linspace(view_min, view_max, nticks))
    else:
        start = np.ceil((view_min - eps) / tick_interval) * tick_interval
        ticks = list(np.arange(start, view_max + eps, tick_interval))
        ticks.extend([view_min, view_max])

    # Keep the endpoints of each compressed interval.
    for step in steps:
        gap_start, gap_end = step[0], step[1]
        ticks.extend([gap_start, gap_end])

    filtered = []
    for t in ticks:
        t = float(t)
        if t < view_min - eps or t > view_max + eps:
            continue
        inside_compressed_gap = any((step[0] + eps < t < step[1] - eps) for step in steps)
        if not inside_compressed_gap:
            filtered.append(t)

    filtered = sorted(filtered)
    unique = []
    for t in filtered:
        if not unique or abs(t - unique[-1]) > eps:
            unique.append(t)
    return unique


def _set_real_energy_ticks(axes, transform_y, view_min, view_max, steps=None, tick_interval=None, nticks=8):
    ticks = _make_real_energy_ticks(view_min, view_max, tick_interval=tick_interval, steps=steps, nticks=nticks)
    tick_pos = [transform_y(float(t)) for t in ticks]
    tick_lab = [f'{float(t):.6g}' for t in ticks]
    for ax in axes:
        ax.set_yticks(tick_pos)
        ax.set_yticklabels(tick_lab)


def _draw_compression_breaks(ax, transform_y, steps, view_min, view_max, width=0.035, height=0.012):
    # Draw paired slash marks on the left y-axis at compressed intervals.
    if not steps:
        return

    y_min_t = transform_y(view_min)
    y_max_t = transform_y(view_max)
    y_span_t = y_max_t - y_min_t
    if y_span_t <= 0:
        return

    for step in steps:
        gap_start, gap_end = step[0], step[1]
        y0 = transform_y(gap_start)
        y1 = transform_y(gap_end)
        y_mid = (y0 + y1) / 2

        # Use two parallel slashes to mark each compressed interval.
        slash_len = y_span_t * height
        slash_sep = slash_len * 0.9
        x0, x1 = -width, width
        for yc in (y_mid - slash_sep / 2, y_mid + slash_sep / 2):
            ax.plot([x0, x1], [yc - slash_len / 2, yc + slash_len / 2],
                    transform=ax.get_yaxis_transform(),
                    color='k', linewidth=1.0, clip_on=False, solid_capstyle='butt')


def _reference_db_for_labels(dbnames, sum_table):
    nonzero_counts = (sum_table.drop('category', axis=1) != 0).sum()
    return nonzero_counts.idxmax()


def _plot_energy_view_on_axes(dbnames, lenergy, sum_table, unit, view_min, view_max, axes,
                              show_labels=True, energy_boundary=None, show_boundary_line=True,
                              cut_big_gap=True, gap_threshold=5.0, display_gap=2.0,
                              shadow_alpha=0.35, line_width=1.1, n_colors=25,
                              ytick_interval=None, draw_compress_breaks=True,
                              break_mark_width=0.035, break_mark_height=0.012,
                              show_ylabel=True, title=None):
    axes = list(axes) if hasattr(axes, '__len__') else [axes]

    palette = sns.color_palette('husl', max(n_colors, len(sum_table) - 1))
    colors = palette.as_hex()
    transform_y, steps = _make_gap_compressor(dbnames, lenergy, sum_table, unit, view_min, view_max,
                                              cut_big_gap=cut_big_gap,
                                              gap_threshold=gap_threshold,
                                              display_gap=display_gap)

    for i, dname in enumerate(dbnames):
        data = lenergy[i]
        col = unit + str(i + 1)
        ax = axes[i]

        for j in range(len(sum_table) - 1):
            category = sum_table.loc[j, 'category']
            if int(sum_table.loc[j, dname]) == 0:
                continue

            cat_data = _get_category_data(data, category, sum_table, col)
            ys = pd.to_numeric(cat_data[col], errors='coerce').dropna().sort_values()
            if len(ys) == 0:
                continue

            y0, y1 = float(ys.min()), float(ys.max())
            y0_clip, y1_clip = max(y0, view_min), min(y1, view_max)
            if y0_clip <= y1_clip:
                ax.axhspan(transform_y(y0_clip), transform_y(y1_clip),
                           color=colors[j % len(colors)], alpha=shadow_alpha, xmin=0.05, xmax=0.95)

            ys_show = ys[(ys >= view_min) & (ys <= view_max)]
            if len(ys_show) > 0:
                ax.hlines(transform_y(ys_show), 0, 1, colors=colors[j % len(colors)], linewidth=line_width)

        if show_boundary_line and energy_boundary is not None and view_min <= energy_boundary <= view_max:
            ax.axhline(transform_y(energy_boundary), color='k', linestyle=':', linewidth=1.0)

        ax.set_xticks([0.5])
        ax.set_xticklabels([dname], rotation=0)
        if i == 0 and show_ylabel:
            ax.set_ylabel('Energy (' + unit + ')')
        else:
            ax.tick_params(left=False, labelleft=False)
            ax.set_ylabel('')
            ax.spines['left'].set_visible(False)

        if title and i == len(dbnames) // 2:
            ax.set_title(title, fontsize=11)

    if show_labels:
        ref_db = _reference_db_for_labels(dbnames, sum_table)
        ref_i = dbnames.index(ref_db)
        ref_data = lenergy[ref_i]
        ref_col = unit + str(ref_i + 1)
        label_rows = sum_table[(sum_table[ref_db] != 0) & (sum_table['category'] != 'total')]

        for row_idx, row in label_rows.iterrows():
            category = row['category']
            cat_data = _get_category_data(ref_data, category, sum_table, ref_col)
            ys = pd.to_numeric(cat_data[ref_col], errors='coerce').dropna()
            if len(ys) == 0:
                continue
            y0, y1 = float(ys.min()), float(ys.max())
            y0_clip, y1_clip = max(y0, view_min), min(y1, view_max)
            if y0_clip <= y1_clip:
                y_text = (y0_clip + y1_clip) / 2
                axes[-1].text(1.1, transform_y(y_text), category, va='center', ha='left',
                              color=colors[row_idx % len(colors)], fontsize=12)

    y_min_t = transform_y(view_min)
    y_max_t = transform_y(view_max)
    for ax in axes:
        ax.set_ylim(y_min_t, y_max_t)
    _set_real_energy_ticks(axes, transform_y, view_min, view_max, steps=steps, tick_interval=ytick_interval)

    if draw_compress_breaks:
        _draw_compression_breaks(axes[0], transform_y, steps, view_min, view_max,
                                 width=break_mark_width, height=break_mark_height)

    return steps


def plot_energy_view(dbnames, lenergy, sum_table, unit, view_min, view_max, filename, title='',
                     show_labels=True, energy_boundary=None, show_boundary_line=True,
                     cut_big_gap=True, gap_threshold=5.0, display_gap=2.0,
                     figsize=None, shadow_alpha=0.35, line_width=1.1, n_colors=25, dpi=200,
                     ytick_interval=None, draw_compress_breaks=True, break_mark_width=0.035,
                     break_mark_height=0.012):
    if figsize is None:
        figsize = (len(dbnames) * 2.2, 8 if show_labels else 10)

    fig, axes = plt.subplots(1, len(dbnames), figsize=figsize, sharey=True)
    axes = list(axes) if hasattr(axes, '__len__') else [axes]

    steps = _plot_energy_view_on_axes(dbnames, lenergy, sum_table, unit, view_min, view_max, axes,
                                      show_labels=show_labels, energy_boundary=energy_boundary,
                                      show_boundary_line=show_boundary_line, cut_big_gap=cut_big_gap,
                                      gap_threshold=gap_threshold, display_gap=display_gap,
                                      shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors,
                                      ytick_interval=ytick_interval, draw_compress_breaks=draw_compress_breaks,
                                      break_mark_width=break_mark_width, break_mark_height=break_mark_height,
                                      show_ylabel=True, title=title)

    plt.subplots_adjust(wspace=-0.01)
    if show_labels:
        fig.subplots_adjust(left=0.18, right=0.75, top=0.94, bottom=0.06)
    else:
        fig.subplots_adjust(left=0.18, right=0.95, top=0.94, bottom=0.06)

    fig.savefig(filename, dpi=dpi, bbox_inches='tight')
    print('saved:', filename)
    plt.show()
    return fig, axes, steps


def _get_three_view_ranges(dbnames, lenergy, unit, energy_boundary):
    all_vals = _all_energy_values(dbnames, lenergy, unit)
    all_vals = all_vals[~np.isnan(all_vals)]
    if len(all_vals) == 0:
        raise ValueError('No valid energy values were found')

    global_min = float(np.min(all_vals))
    global_max = float(np.max(all_vals))

    below = all_vals[all_vals <= energy_boundary]
    above = all_vals[all_vals >= energy_boundary]

    low_min = 0.0
    low_max = float(np.max(below)) if len(below) > 0 else float(energy_boundary)
    high_min = float(np.min(above)) if len(above) > 0 else float(energy_boundary)
    high_max = global_max

    return {
        'full': (global_min, global_max),
        'low': (low_min, low_max),
        'high': (high_min, high_max)
    }


def plot_combined_three_views(dbnames, lenergy, sum_table, unit, output_dir, energy_boundary,
                              cut_big_gap=True, gap_threshold=5.0, display_gap=2.0,
                              figsize_combined=(15, 5.5), shadow_alpha=0.35, line_width=1.1,
                              n_colors=25, dpi=200, show_boundary_line=True, ytick_interval=None,
                              draw_compress_breaks=True, break_mark_width=0.035, break_mark_height=0.012,
                              output_format='pdf'):
    ranges = _get_three_view_ranges(dbnames, lenergy, unit, energy_boundary)
    view_specs = [
        ('Full', ranges['full'][0], ranges['full'][1], False),
        ('Low', ranges['low'][0], ranges['low'][1], True),
        ('High', ranges['high'][0], ranges['high'][1], True),
    ]

    ndb = len(dbnames)
    fig, all_axes = plt.subplots(1, ndb * 3, figsize=figsize_combined, sharey=False)
    all_axes = np.array(all_axes).reshape(3, ndb)

    for view_idx, (title, view_min, view_max, show_labels) in enumerate(view_specs):
        axes = list(all_axes[view_idx])
        _plot_energy_view_on_axes(dbnames, lenergy, sum_table, unit, view_min, view_max, axes,
                                  show_labels=show_labels, energy_boundary=energy_boundary,
                                  show_boundary_line=show_boundary_line, cut_big_gap=cut_big_gap,
                                  gap_threshold=gap_threshold, display_gap=display_gap,
                                  shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors,
                                  ytick_interval=ytick_interval, draw_compress_breaks=draw_compress_breaks,
                                  break_mark_width=break_mark_width, break_mark_height=break_mark_height,
                                  show_ylabel=True, title=title)

        # Restore the left axis at the start of each view group.
        if view_idx > 0:
            axes[0].spines['left'].set_visible(True)
            axes[0].tick_params(left=True, labelleft=True)

    fig.subplots_adjust(left=0.055, right=0.94, top=0.92, bottom=0.09, wspace=0.10)
    combined_path = os.path.join(output_dir, f'Evsdb_0_three_views_combined.{output_format}')
    fig.savefig(combined_path, dpi=dpi, bbox_inches='tight')
    print('saved:', combined_path)
    plt.show()
    return fig, all_axes


def plot_three_energy_views(dbnames, lenergy, sum_table, unit, output_dir, energy_boundary,
                            cut_big_gap=True, gap_threshold=5.0, display_gap=2.0,
                            figsize_full=None, figsize_split=None, figsize_combined=(15, 5.5),
                            shadow_alpha=0.35, line_width=1.1, n_colors=25, dpi=200,
                            show_boundary_line=True, ytick_interval=None, draw_compress_breaks=True,
                            break_mark_width=0.035, break_mark_height=0.012,
                            output_format='pdf', make_combined=True):
    ranges = _get_three_view_ranges(dbnames, lenergy, unit, energy_boundary)
    global_min, global_max = ranges['full']
    low_min, low_max = ranges['low']
    high_min, high_max = ranges['high']

    if figsize_full is None:
        figsize_full = (len(dbnames) * 2.2, 10)
    if figsize_split is None:
        figsize_split = (len(dbnames) * 2.2, 7)

    os.makedirs(output_dir, exist_ok=True)

    print('full range:', global_min, global_max)
    print('low range:', low_min, low_max)
    print('high range:', high_min, high_max)

    plot_energy_view(dbnames, lenergy, sum_table, unit, global_min, global_max,
                     os.path.join(output_dir, f'Evsdb_1_full_no_labels.{output_format}'),
                     title=None, show_labels=False, energy_boundary=energy_boundary,
                     show_boundary_line=show_boundary_line, cut_big_gap=cut_big_gap,
                     gap_threshold=gap_threshold, display_gap=display_gap, figsize=figsize_full,
                     shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors, dpi=dpi,
                     ytick_interval=ytick_interval, draw_compress_breaks=draw_compress_breaks,
                     break_mark_width=break_mark_width, break_mark_height=break_mark_height)

    plot_energy_view(dbnames, lenergy, sum_table, unit, low_min, low_max,
                     os.path.join(output_dir, f'Evsdb_2_low_with_labels.{output_format}'),
                     title=None, show_labels=True, energy_boundary=energy_boundary,
                     show_boundary_line=show_boundary_line, cut_big_gap=cut_big_gap,
                     gap_threshold=gap_threshold, display_gap=display_gap, figsize=figsize_split,
                     shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors, dpi=dpi,
                     ytick_interval=ytick_interval, draw_compress_breaks=draw_compress_breaks,
                     break_mark_width=break_mark_width, break_mark_height=break_mark_height)

    plot_energy_view(dbnames, lenergy, sum_table, unit, high_min, high_max,
                     os.path.join(output_dir, f'Evsdb_3_high_with_labels.{output_format}'),
                     title=None, show_labels=True, energy_boundary=energy_boundary,
                     show_boundary_line=show_boundary_line, cut_big_gap=cut_big_gap,
                     gap_threshold=gap_threshold, display_gap=display_gap, figsize=figsize_split,
                     shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors, dpi=dpi,
                     ytick_interval=ytick_interval, draw_compress_breaks=draw_compress_breaks,
                     break_mark_width=break_mark_width, break_mark_height=break_mark_height)

    if make_combined:
        plot_combined_three_views(dbnames, lenergy, sum_table, unit, output_dir, energy_boundary,
                                  cut_big_gap=cut_big_gap, gap_threshold=gap_threshold,
                                  display_gap=display_gap, figsize_combined=figsize_combined,
                                  shadow_alpha=shadow_alpha, line_width=line_width, n_colors=n_colors,
                                  dpi=dpi, show_boundary_line=show_boundary_line,
                                  ytick_interval=ytick_interval,
                                  draw_compress_breaks=draw_compress_breaks,
                                  break_mark_width=break_mark_width,
                                  break_mark_height=break_mark_height,
                                  output_format=output_format)


In [ ]:
# Generate the three individual PDF views and optionally the combined view.
lenergy, sum_table, output_dir = prepare_energy_data(ion, dbnames, unit)

plot_three_energy_views(dbnames, lenergy, sum_table, unit, output_dir, energy_boundary,
                        cut_big_gap=cut_big_gap,
                        gap_threshold=compress_gap_threshold,
                        display_gap=compressed_gap_display,
                        figsize_full=figsize_full,
                        figsize_split=figsize_split,
                        figsize_combined=figsize_combined,
                        shadow_alpha=shadow_alpha,
                        line_width=line_width,
                        n_colors=n_colors,
                        dpi=dpi,
                        show_boundary_line=show_boundary_line,
                        ytick_interval=ytick_interval,
                        draw_compress_breaks=draw_compress_breaks,
                        break_mark_width=break_mark_width,
                        break_mark_height=break_mark_height,
                        output_format=output_format,
                        make_combined=False)

print('Output directory:', output_dir)
